# CHURN PIPELINE - PHIÊN BẢN CẢI TIẾN V2 (CALIBRATED & REGULARIZED)

Notebook này đã được tối ưu hóa để giải quyết triệt để các vấn đề:
1. **Loại bỏ rò rỉ dữ liệu (Data Leakage)**: Không dùng Segment trong huấn luyện.
2. **Xử lý Overfitting**: Giảm độ phức tạp mô hình và tăng cường Regularization.
3. **Hiệu chỉnh xác suất (Calibration)**: Đảm bảo ChurnProb phản ánh đúng thực tế.
4. **Tối ưu ngưỡng (F2-Tuning)**: Ưu tiên bắt trọn khách hàng Churn (giảm False Negative).

In [ ]:
# STEP 0 - Import thư viện
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing import Dict, List, Tuple
from IPython.display import display, Image

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, roc_auc_score, confusion_matrix, 
    precision_recall_fscore_support, fbeta_score, precision_recall_curve
)
from sklearn.model_selection import TimeSeriesSplit, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except:
    HAS_XGBOOST = False

print('HAS_XGBOOST =', HAS_XGBOOST)
print('Setup complete.')

In [ ]:
# STEP 1 - Cấu hình chạy notebook
INPUT_FILE = 'ver1.xlsx'
DATA_SHEET = 'Data Model'
LOOKBACK_DAYS = 90
AS_OF_TRAIN = pd.Timestamp('2023-10-31')

MODE = 'full' # 'full' | 'backtest' | 'forecast_dec'

OUTPUT_SCORING = 'churn_list.xlsx'
OUTPUT_RATE_REPORT = 'churn_rate_report.xlsx'
OUTPUT_COMPARISON = 'churn_model_comparison.xlsx'
OUTPUT_DEEPDIVE = 'churn_deepdive_data.xlsx'

print('MODE =', MODE)

In [ ]:
# STEP 2 - Load và làm sạch dữ liệu
def load_and_clean_data(path: str, sheet: str) -> pd.DataFrame:
    df = pd.read_excel(path, sheet_name=sheet)
    df['Ngày Xuất'] = pd.to_datetime(df['Ngày Xuất'], errors='coerce')
    for col in ['Doanh Thu', 'Lợi Nhuận', '% SL Khuyến mãi']:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
    df['Khách hàng'] = df['Khách hàng'].astype(str).str.strip()
    return df.dropna(subset=['Ngày Xuất'])

df = load_and_clean_data(INPUT_FILE, DATA_SHEET)
print(f"Loaded {len(df)} transactions.")

In [ ]:
# STEP 3 - Utility Functions (Tối ưu hóa: Logistic Regression + Random Forest)
def build_features(df: pd.DataFrame, as_of_date: pd.Timestamp, lookback_days: int) -> pd.DataFrame:
    start = as_of_date - pd.Timedelta(days=lookback_days)
    hist = df[(df['Ngày Xuất'] > start) & (df['Ngày Xuất'] <= as_of_date)].copy()
    if hist.empty: return pd.DataFrame()
    agg = hist.groupby('Khách hàng').agg(first_purchase=('Ngày Xuất', 'min'), last_purchase=('Ngày Xuất', 'max'), Frequency=('Khách hàng', 'size'), Revenue=('Doanh Thu', 'sum'), Profit=('Lợi Nhuận', 'sum'), PromoRate=('% SL Khuyến mãi', 'mean'))
    agg['Recency'] = (as_of_date - agg['last_purchase']).dt.days
    agg['DaysSinceFirst'] = (as_of_date - agg['first_purchase']).dt.days
    agg['AOV'] = np.where(agg['Frequency'] > 0, agg['Revenue'] / agg['Frequency'], 0)
    agg['Margin'] = np.where(agg['Revenue'] != 0, agg['Profit'] / agg['Revenue'], 0)
    recent = hist[hist['Ngày Xuất'] > as_of_date - pd.Timedelta(days=30)]
    older = hist[(hist['Ngày Xuất'] <= as_of_date - pd.Timedelta(days=30)) & (hist['Ngày Xuất'] > as_of_date - pd.Timedelta(days=60))]
    agg['Trend'] = (recent.groupby('Khách hàng')['Doanh Thu'].sum() - older.groupby('Khách hàng')['Doanh Thu'].sum()).reindex(agg.index).fillna(0)
    hist['Month'] = hist['Ngày Xuất'].dt.to_period('M')
    agg['ActiveMonths'] = hist.groupby('Khách hàng')['Month'].nunique().reindex(agg.index).fillna(0)
    seg_lookup = df[df['Ngày Xuất'] <= as_of_date].sort_values('Ngày Xuất').groupby('Khách hàng')['Segment'].last()
    agg['Segment'] = seg_lookup.reindex(agg.index).fillna('Unknown')
    return agg.reset_index()[['Khách hàng', 'Segment', 'Recency', 'Frequency', 'AOV', 'PromoRate', 'Margin', 'Trend', 'DaysSinceFirst', 'ActiveMonths']]

def label_churn_window(df: pd.DataFrame, customers: pd.Series, start: pd.Timestamp, end: pd.Timestamp) -> pd.Series:
    active = set(df[(df['Ngày Xuất'] >= start) & (df['Ngày Xuất'] <= end)]['Khách hàng'].unique())
    return (~customers.isin(active)).astype(int)

def preprocess(scale_numeric: bool) -> ColumnTransformer:
    num_cols = ['Recency', 'Frequency', 'AOV', 'PromoRate', 'Margin', 'Trend', 'DaysSinceFirst', 'ActiveMonths']
    num_steps = [('imputer', SimpleImputer(strategy='constant', fill_value=0.0))]
    if scale_numeric: num_steps.append(('scaler', StandardScaler()))
    return ColumnTransformer([('num', Pipeline(num_steps), num_cols)], remainder='drop')

def get_models() -> Dict[str, Pipeline]:
    from sklearn.calibration import CalibratedClassifierCV
    from sklearn.linear_model import LogisticRegression
    from sklearn.ensemble import RandomForestClassifier
    
    # 1. Logistic Regression (Ổn định cơ sở)
    lr_base = Pipeline([('p', preprocess(True)), ('c', LogisticRegression(C=0.5, class_weight={0:1, 1:2}, max_iter=3000, random_state=42))])
    
    # 2. Random Forest (Bắt quy luật phi tuyến, đã Calibration)
    rf_base = Pipeline([('p', preprocess(False)), ('c', RandomForestClassifier(n_estimators=150, max_depth=4, min_samples_leaf=5, class_weight={0:1,1:2}, random_state=42))])
    
    models = {
        'logistic_regression': lr_base,
        'random_forest': CalibratedClassifierCV(rf_base, method='sigmoid', cv=5)
    }
    return models

def safe_write_excel(path: Path, write_fn):
    try:
        with pd.ExcelWriter(path, engine='openpyxl') as writer: write_fn(writer)
        return path
    except PermissionError:
        fallback = path.with_name(f'{path.stem}_new{path.suffix}')
        with pd.ExcelWriter(fallback, engine='openpyxl') as writer: write_fn(writer)
        return fallback

def action_rule(row: pd.Series) -> str:
    if row.get('Segment') == 'Khách hàng VIP': return 'Gặp mặt trực tiếp'
    if row['Recency'] > 60: return 'Gọi ngay'
    if row['Recency'] > 30 or row['ChurnProb'] > 0.8: return 'Zalo offer'
    return 'Theo dõi định kỳ'
print('Utilities ready (LR + RF).')

In [ ]:
# STEP 4A - Benchmark TimeSeries CV
def run_compare(df: pd.DataFrame):
    train = build_features(df, AS_OF_TRAIN, LOOKBACK_DAYS)
    train['Churn30'] = label_churn_window(df, train['Khách hàng'], AS_OF_TRAIN + pd.Timedelta(days=1), AS_OF_TRAIN + pd.offsets.MonthEnd(1))
    X, y = train.drop(columns=['Khách hàng', 'Segment', 'Churn30']), train['Churn30'].values
    cv = TimeSeriesSplit(n_splits=3)
    rows = []
    for name, model in get_models().items():
        scores = cross_validate(model, X, y, cv=cv, scoring=['roc_auc', 'average_precision'])
        model.fit(X, y)
        rows.append({'model': name, 'cv_auc': np.mean(scores['test_roc_auc']), 'train_auc': roc_auc_score(y, model.predict_proba(X)[:, 1])})
    return pd.DataFrame(rows).sort_values('cv_auc', ascending=False)

if MODE in ['full', 'compare']:
    display(run_compare(df))

In [ ]:
# STEP 4B - Backtest, Ensemble Weights & Optimal Threshold
from sklearn.metrics import recall_score, precision_score

def find_optimal_threshold(prob, y_true, max_urgent=50, target_recall=0.80):
    """Tìm ngưỡng cân bằng giữa F2-Score và Năng lực Sales"""
    pre, rec, ths = precision_recall_curve(y_true, prob)
    f2 = (5 * pre * rec) / (4 * pre + rec + 1e-8)
    f2_t = float(ths[np.argmax(f2)])
    
    best_t = f2_t
    best_precision = 0.0
    
    for t in np.arange(f2_t, 0.55, 0.01):
        pred = (prob >= t).astype(int)
        r = recall_score(y_true, pred, zero_division=0)
        p = precision_score(y_true, pred, zero_division=0)
        n = pred.sum()
        
        if r >= target_recall and n <= max_urgent and p > best_precision:
            best_precision = p
            best_t = float(t)
            
    print(f"F2 Threshold: {f2_t:.3f} | Optimal Threshold: {best_t:.3f} | Precision at Optimal: {best_precision:.3f}")
    return best_t, f2_t

def run_backtest_final(df: pd.DataFrame):
    f_tr = build_features(df, pd.Timestamp('2023-09-30'), LOOKBACK_DAYS)
    y_tr = label_churn_window(df, f_tr['Khách hàng'], pd.Timestamp('2023-10-01'), pd.Timestamp('2023-10-31'))
    f_ts = build_features(df, pd.Timestamp('2023-10-31'), LOOKBACK_DAYS)
    y_ts = label_churn_window(df, f_ts['Khách hàng'], pd.Timestamp('2023-11-01'), pd.Timestamp('2023-11-30'))
    feat_cols = ['Recency', 'Frequency', 'AOV', 'PromoRate', 'Margin', 'Trend', 'DaysSinceFirst', 'ActiveMonths']
    
    models = get_models()
    probs_dict = {}
    model_aucs = {}
    
    for name, m in models.items():
        m.fit(f_tr[feat_cols], y_tr.values)
        p = m.predict_proba(f_ts[feat_cols])[:, 1]
        probs_dict[name] = p
        model_aucs[name] = roc_auc_score(y_ts, p)
        
    # Weighted Ensemble từ Backtest AUC
    total_auc = sum(model_aucs.values())
    weights = {name: auc / total_auc for name, auc in model_aucs.items()}
    ens_prob = sum(probs_dict[name] * weights[name] for name in probs_dict)
    
    # Lưu Weights cho Forecast
    global BACKTEST_WEIGHTS
    BACKTEST_WEIGHTS = weights
    print("Backtest Weights saved:", {k: f"{v:.3f}" for k,v in weights.items()})
    
    final_t, f2_t = find_optimal_threshold(ens_prob, y_ts, max_urgent=55, target_recall=0.72)
    
    # Results & Visualization Data
    y_pred = (ens_prob >= final_t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_ts, y_pred, labels=[0, 1]).ravel()
    global BACKTEST_CM_DATA; BACKTEST_CM_DATA = {'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)}
    
    safe_write_excel(Path(OUTPUT_DEEPDIVE), lambda w: (
        pd.DataFrame([{'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp), 'optimal_t': final_t}]).to_excel(w, sheet_name='nov_summary', index=False)
    ))
    
    print(f"Backtest Nov - AUC: {roc_auc_score(y_ts, ens_prob):.3f} | Optimal T: {final_t:.3f}")
    print(f"Confusion Matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}")
    return final_t

if MODE in ['full', 'backtest']:
    BEST_THRESHOLD = run_backtest_final(df)


In [ ]:
# STEP 4E - Heatmap Confusion Matrix (BẢN FIX TRIỆT ĐỂ)
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Ưu tiên lấy từ bộ nhớ, nếu không có mới đọc file
data = globals().get('BACKTEST_CM_DATA', None)
if data is None and os.path.exists(OUTPUT_DEEPDIVE):
    try:
        data = pd.read_excel(OUTPUT_DEEPDIVE, sheet_name='nov_summary').iloc[0].to_dict()
    except: pass

if data:
    plt.figure(figsize=(7, 6))
    # Ép kiểu int và dùng annot=True để hiện số
    matrix = [[int(data['tn']), int(data['fp'])], [int(data['fn']), int(data['tp'])]]
    sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues', 
                annot_kws={'size': 20, 'weight': 'bold', 'color': 'black'})
    plt.title('MA TRẬN NHẦM LẪN THÁNG 11', fontsize=15)
    plt.xlabel('DỰ BÁO'); plt.ylabel('THỰC TẾ')
    plt.show()
else:
    print('LỖI: Không tìm thấy dữ liệu. Hãy chạy STEP 4B trước!')

In [ ]:
# STEP 4C - Forecast December (Final V3)
def run_forecast_final(df: pd.DataFrame, threshold: float):
    f1 = build_features(df, pd.Timestamp('2023-09-30'), LOOKBACK_DAYS)
    l1 = label_churn_window(df, f1['Khách hàng'], pd.Timestamp('2023-10-01'), pd.Timestamp('2023-10-31'))
    f2 = build_features(df, pd.Timestamp('2023-10-31'), LOOKBACK_DAYS)
    l2 = label_churn_window(df, f2['Khách hàng'], pd.Timestamp('2023-11-01'), pd.Timestamp('2023-11-30'))
    
    train_multi = pd.concat([f1, f2], ignore_index=True)
    y_train = pd.concat([l1, l2], ignore_index=True).values
    
    f_nov = build_features(df, pd.Timestamp('2023-11-30'), LOOKBACK_DAYS)
    feat_cols = ['Recency', 'Frequency', 'AOV', 'PromoRate', 'Margin', 'Trend', 'DaysSinceFirst', 'ActiveMonths']
    
    models = get_models()
    probs_dict = {}
    
    for name, m in models.items():
        m.fit(train_multi[feat_cols], y_train)
        probs_dict[name] = m.predict_proba(f_nov[feat_cols])[:, 1]

    # Dùng Weights từ Backtest
    saved_weights = globals().get('BACKTEST_WEIGHTS', None)
    if saved_weights:
        weights = saved_weights
        print("Sử dụng trọng số từ Backtest:", {k: f"{v:.3f}" for k,v in weights.items()})
    else:
        weights = {name: 1.0/len(models) for name in models}
        print("Cảnh báo: Không tìm thấy BACKTEST_WEIGHTS, dùng Equal Weights.")
        
    f_nov['ChurnProb'] = sum(probs_dict[name] * weights.get(name, 1.0/len(models)) for name in probs_dict)
    
    # Thêm RiskScore 1-10
    f_nov['RiskScore'] = (f_nov['ChurnProb'] * 10).clip(1, 10).round(1)
    
    # Phân loại rủi ro rõ ràng hơn bằng pd.cut
    f_nov['Mức độ'] = pd.cut(
        f_nov['ChurnProb'],
        bins=[-0.1, 0.35, 0.55, 0.75, 1.1],
        labels=['Thấp', 'Trung bình', 'Cao', 'Khẩn cấp']
    )
    
    f_nov['Hành động'] = f_nov.apply(action_rule, axis=1)
    
    out = f_nov[['Khách hàng', 'Segment', 'Recency', 'Frequency', 'AOV', 'Margin', 'RiskScore', 'ChurnProb', 'Mức độ', 'Hành động']]
    safe_write_excel(Path(OUTPUT_SCORING), lambda w: out.to_excel(w, sheet_name='churn_scoring', index=False))
    print('Forecast Dec saved to', OUTPUT_SCORING)
    return out

if MODE in ['full', 'forecast_dec']:
    scored_df = run_forecast_final(df, globals().get('BEST_THRESHOLD', 0.40))
    display(scored_df.head(10))

In [ ]:
# STEP 5 - Rate Report
if 'scored_df' in globals():
    report = scored_df.groupby('Segment').agg(count=('Khách hàng','count'), prob=('ChurnProb','mean')).sort_values('prob', ascending=False)
    display(report)